# Lesson 5: RAG for Agents

## Activity 01: Build a Knowledge-Aware Agent

In Week 1 you built an agent that could use tools like a calculator, search and a weather API. In this activity you give your agent a new tool: **a knowledge base it can search**.

By the end, your agent will:

- decide for itself whether a question needs the knowledge base,
- search for the most relevant documents,
- answer using what it found, and name its sources,
- admit when the knowledge base doesn't cover a question.

## Learning objectives

By the end of this activity, you should be able to:

1. Explain why agents need external knowledge.
2. Describe the three steps of RAG: retrieve, augment, generate.
3. Explain what embeddings are and how semantic search works.
4. Store and search documents with ChromaDB.
5. Turn retrieval into a tool the agent can call.
6. Build an agent that decides when to retrieve.

**Time:** about 90 minutes
**You'll need:** a Gemini API key, and the Week 1 tool-calling activity completed.

> **How to use this notebook:** run the cells from top to bottom. Read each explanation before running the code below it. Cells marked **Try it** invite you to change something and run again.

## 1. Why agents need external knowledge

A language model only knows what was in its training data. That causes three problems:

| Problem | Example |
|---|---|
| **It doesn't know private information** | Your company's policies, your course notes, your product manual |
| **Its knowledge stops at a cutoff date** | Anything that changed after the model was trained |
| **It may make things up** | When it doesn't know, it can produce a confident but wrong answer, called a *hallucination* |

Think of it like an exam. A closed-book exam relies only on what you remember. An **open-book exam** lets you look things up, so your answers are more accurate and you can point to where you found them.

Giving an agent external knowledge turns every question into an open-book exam.

## 2. What is RAG?

**RAG** stands for **Retrieval-Augmented Generation**. It has three steps:

```text
Question
   ↓
1. RETRIEVE   find the documents most relevant to the question
   ↓
2. AUGMENT    add those documents to what the model sees
   ↓
3. GENERATE   the model writes an answer based on those documents
   ↓
Answer (with sources)
```

### RAG for agents

In basic RAG, the system searches **every time**, even for "Hi, how are you?"

In **agentic RAG**, retrieval is a **tool**, just like the calculator in Week 1. The agent decides:

- **whether** it needs to search,
- **what** to search for,
- **whether to search again** if the first results weren't enough.

| | Basic RAG | Agentic RAG (this activity) |
|---|---|---|
| When does it search? | Always | Only when the agent decides it's needed |
| Who writes the search query? | The user's question is used as-is | The agent writes its own query |
| Can it search twice? | No | Yes |

## 3. Setup

We'll use four packages:

| Package | What it does |
|---|---|
| `google-genai` | Talks to Gemini (same as Week 1) |
| `sentence-transformers` | Turns text into embeddings, free and on your machine |
| `chromadb` | Stores embeddings and searches them |

Run the cell below. It may take a minute.

In [ ]:
!pip -q install google-genai sentence-transformers chromadb

### Add your Gemini API key

Create a key in [Google AI Studio](https://aistudio.google.com) if you don't have one. Enter it when prompted. It won't be shown on screen or saved in the notebook.

**Never paste your API key directly into a code cell**, especially in a notebook you'll share or submit.

In [ ]:
import os
import getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

print("API key loaded.")

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

MODEL = "gemini-3.8-flash"

print("Gemini client is ready. Model:", MODEL)

## 4. Embeddings and semantic search

To find relevant documents, we need a way to compare **meaning**, not just words.

### Keyword search vs semantic search

Suppose the knowledge base contains: *"A vehicle needs regular servicing."*

- **Keyword search** for "car maintenance" finds nothing, because no words match.
- **Semantic search** finds it, because "car" ≈ "vehicle" and "maintenance" ≈ "servicing".

### What is an embedding?

An **embedding** is a list of numbers that represents the meaning of a piece of text. A model turns each text into the same-length list, and **texts with similar meanings get similar numbers**.

```text
"car maintenance"          →  [0.12, -0.48, 0.33, ... ]   (384 numbers)
"vehicle servicing"        →  [0.10, -0.45, 0.30, ... ]   ← very close
"chocolate cake recipe"    →  [-0.62, 0.21, -0.08, ...]   ← far away
```

We measure closeness with **cosine similarity**: a score where **1** means the same meaning and values near **0** mean unrelated.

We'll use `all-MiniLM-L6-v2`, a small, free embedding model. The first time you load it, it downloads about 90 MB.

In [ ]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "car maintenance",
    "vehicle servicing",
    "chocolate cake recipe",
]

vectors = embedder.encode(sentences)

print("Each embedding has", vectors.shape[1], "numbers.\n")
print("First 5 numbers of 'car maintenance':", vectors[0][:5].round(3), "\n")

scores = util.cos_sim(vectors, vectors)
print(f"car maintenance vs vehicle servicing:     {scores[0][1]:.2f}")
print(f"car maintenance vs chocolate cake recipe: {scores[0][2]:.2f}")

The first pair shares **no words**, yet scores much higher than the second. That is semantic search.

> **Try it:** change the sentences above to your own examples. Can you find two sentences with different words but a high score?

## 5. Create the knowledge base

Our knowledge base is 8 short documents about Agentic AI, the topics from this course so far. Each one has:

- an `id`, so we can track which document was used,
- a `title`, so the agent can name its sources,
- the `text` itself.

Real knowledge bases often contain long documents, which are split into smaller pieces called **chunks** before they're embedded. Our documents are short, so each one is a single chunk.

In [ ]:
KNOWLEDGE_BASE = [
    {
        "id": "kb-01",
        "title": "What is an AI agent?",
        "text": (
            "An AI agent is a system that uses a language model to pursue a goal by deciding what to do next. "
            "Unlike a chatbot, which only replies to messages, an agent can take actions with tools, look at the "
            "results, and choose its next step. A workflow follows a fixed path written by a developer; an agent "
            "chooses its own path. Most real agents are assisted: a human approves important actions such as "
            "payments or sending emails."
        ),
    },
    {
        "id": "kb-02",
        "title": "Tools and function calling",
        "text": (
            "A tool is a function an agent can ask to use, such as a calculator, a web search or an API. The model "
            "never runs the tool itself. It returns a function call containing the tool name and arguments; your "
            "Python code runs the function and sends the result back. Each tool is described with a schema: a name, "
            "a description and parameters. The model chooses tools using only these descriptions, so clear "
            "descriptions matter more than clever code."
        ),
    },
    {
        "id": "kb-03",
        "title": "The ReAct loop and stopping rules",
        "text": (
            "ReAct stands for Reason and Act. The agent reasons about what it needs, acts by calling a tool, "
            "observes the result, and repeats until it can answer. Every agent loop needs three stopping rules: "
            "stop when the model makes no more tool calls, stop when the step budget runs out, and block repeated "
            "identical calls with a loop guard. Without them, an agent can loop forever and waste money."
        ),
    },
    {
        "id": "kb-04",
        "title": "Agent memory",
        "text": (
            "Short-term memory is the conversation history the agent sends to the model on every turn. It is limited "
            "by the context window, the maximum number of tokens the model can read at once. Long-term memory lives "
            "outside the model, for example in a database, and is retrieved only when needed. Retrieval lets an "
            "agent use far more information than fits in its context window."
        ),
    },
    {
        "id": "kb-05",
        "title": "Retrieval-Augmented Generation (RAG)",
        "text": (
            "RAG has three steps: retrieve relevant documents, augment the prompt with them, and generate an answer "
            "grounded in that text. It reduces hallucination because the model answers from real sources it can "
            "cite. Unlike fine-tuning, RAG does not change the model's weights; to update its knowledge you simply "
            "update the documents. In agentic RAG, retrieval is a tool, and the agent decides when to search and "
            "what to search for."
        ),
    },
    {
        "id": "kb-06",
        "title": "Embeddings and vector databases",
        "text": (
            "An embedding is a list of numbers that represents the meaning of a piece of text. Texts with similar "
            "meanings have embeddings that are close together, even if they share no words, so 'car' and 'vehicle' "
            "end up near each other. Closeness is usually measured with cosine similarity. A vector database such "
            "as ChromaDB stores embeddings and quickly finds the ones closest to a query. This is called semantic search."
        ),
    },
    {
        "id": "kb-07",
        "title": "Multi-agent systems",
        "text": (
            "A multi-agent system splits work between several agents with different roles, such as a planner, a "
            "researcher and a writer. An orchestrator agent assigns tasks and combines the results. Agents pass work "
            "to each other through handoffs. Multiple agents help with large or varied tasks, but they cost more and "
            "are harder to debug, so start with a single agent and add more only when you need them."
        ),
    },
    {
        "id": "kb-08",
        "title": "Agent safety and guardrails",
        "text": (
            "Agents act in the real world, so they need guardrails. Give each agent only the tools it needs, a "
            "principle called least privilege. Ask a human to approve risky actions such as payments, deletions or "
            "sending messages. Treat text that comes back from tools and documents as data, never as instructions; "
            "attackers can hide commands in web pages, an attack called prompt injection. Log every step so you can "
            "audit what the agent did."
        ),
    },
]

print(f"Knowledge base has {len(KNOWLEDGE_BASE)} documents:")
for doc in KNOWLEDGE_BASE:
    print(f"  {doc['id']}  {doc['title']}")

## 6. Generate embeddings

Now we turn every document into an embedding. ChromaDB expects plain Python lists, so we convert with `.tolist()`.

In [ ]:
def embed(texts):
    # Turn a list of texts into a list of embedding vectors.
    return embedder.encode(texts).tolist()


doc_embeddings = embed([doc["text"] for doc in KNOWLEDGE_BASE])

print("Documents embedded:", len(doc_embeddings))
print("Numbers per embedding:", len(doc_embeddings[0]))

## 7. Store and search embeddings with ChromaDB

**ChromaDB** is a vector database: it stores embeddings and quickly finds the ones closest to a query.

A few terms:

| Term | Meaning |
|---|---|
| **Collection** | A named group of documents, like a table in a spreadsheet |
| `"hnsw:space": "cosine"` | Compare embeddings using cosine distance (meaning) |
| **Distance** | ChromaDB returns *distance*, where lower is closer. We convert it: `similarity = 1 - distance` |

We'll keep the database **in memory**, so nothing is saved to disk. We wrap this in a function so you can rebuild the knowledge base later in the challenge.

In [ ]:
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
COLLECTION_NAME = "agentic_ai_kb"


def build_collection(documents):
    # Create a fresh collection and add every document with its embedding.
    try:
        chroma_client.delete_collection(COLLECTION_NAME)   # start fresh, so re-running never adds duplicates
    except Exception:
        pass

    new_collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )
    new_collection.add(
        ids=[doc["id"] for doc in documents],
        documents=[doc["text"] for doc in documents],
        embeddings=embed([doc["text"] for doc in documents]),
        metadatas=[{"title": doc["title"]} for doc in documents],
    )
    return new_collection


collection = build_collection(KNOWLEDGE_BASE)
print("Documents stored in ChromaDB:", collection.count())

### Try a search

Let's search with a question. Notice that the question doesn't use the same words as the document it should find.

In [ ]:
query = "How do I stop an agent from running forever?"

raw = collection.query(
    query_embeddings=embed([query]),
    n_results=3,
)

print("Query:", query, "\n")
for doc_id, metadata, distance in zip(raw["ids"][0], raw["metadatas"][0], raw["distances"][0]):
    print(f"{1 - distance:.2f}  {doc_id}  {metadata['title']}")

The top result should be **The ReAct loop and stopping rules**, even though the query never says "ReAct" or "stopping rules".

ChromaDB returns **lists of lists**, one inner list per query. We sent one query, so we read `[0]`.

> **Try it:** change `query` to a question of your own and re-run the cell.

## 8. Create the `knowledge_search()` tool

Now we wrap the search in a **tool function**, just like the Week 1 tools. It follows the same two rules:

1. **Return a dictionary** the model can read.
2. **Never crash.** If something goes wrong, return `{"error": "..."}`.

It also filters out weak matches. Search always returns the *closest* documents, even if none of them are actually relevant. A **minimum similarity** threshold stops the agent from answering with unrelated text.

In [ ]:
TOP_K = 3              # how many documents to return per search
MIN_SIMILARITY = 0.35  # ignore matches weaker than this


def knowledge_search(query: str) -> dict:
    # Search the knowledge base and return the most relevant documents.
    try:
        raw = collection.query(query_embeddings=embed([query]), n_results=TOP_K)
    except Exception as e:
        return {"error": f"Knowledge search failed: {e}"}

    results = []
    for doc_id, text, metadata, distance in zip(
        raw["ids"][0], raw["documents"][0], raw["metadatas"][0], raw["distances"][0]
    ):
        similarity = round(1 - distance, 3)
        if similarity < MIN_SIMILARITY:
            continue   # too weak to be useful
        results.append({"id": doc_id, "title": metadata["title"], "similarity": similarity, "text": text})

    if not results:
        return {"query": query, "results": [], "note": "No relevant documents found in the knowledge base."}
    return {"query": query, "results": results}

In [ ]:
import json

print("A question the knowledge base covers:")
print(json.dumps(knowledge_search("What is prompt injection?"), indent=2)[:700], "...\n")

print("A question it doesn't cover:")
print(json.dumps(knowledge_search("best chocolate cake recipe"), indent=2))

The first search finds **Agent safety and guardrails**. The second returns no results, with a note. That note is important: it lets the agent tell the user honestly that the knowledge base doesn't cover the question.

## 9. Give the tool to Gemini

As in Week 1, Gemini never sees your Python function. It sees only a **declaration**: the tool's name, description and parameters.

For a retrieval tool, the description does an extra job: it tells Gemini **when to retrieve**. A good description says:

- what the knowledge base contains,
- when to use the tool,
- when *not* to use it.

We also write a **system instruction** that sets the rules for every answer.

> **Note:** Gemini declarations don't include a `"type": "function"` key. That's another provider's format, and the Gemini SDK rejects it.

In [ ]:
knowledge_search_declaration = {
    "name": "knowledge_search",
    "description": (
        "Search the course knowledge base about agentic AI: AI agents, tools and function calling, "
        "the ReAct loop and stopping rules, agent memory, RAG, embeddings and vector databases, "
        "multi-agent systems, and agent safety. Use it for any question about these topics. "
        "Do not use it for greetings, small talk or simple arithmetic."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "A short search phrase for what to look up, e.g. 'stopping rules for agent loops'.",
            }
        },
        "required": ["query"],
    },
}

TOOL_DECLARATIONS = [knowledge_search_declaration]

AVAILABLE_TOOLS = {
    "knowledge_search": knowledge_search,
}

SYSTEM_INSTRUCTION = """You are a helpful assistant for a beginner course on Agentic AI.
You have one tool, knowledge_search, which searches the course knowledge base.
- For questions about agentic AI concepts, call knowledge_search first and answer using only what it returns.
- You may search more than once if a question has several parts.
- For greetings, small talk or simple arithmetic, answer directly without searching.
- If the search returns no results, or the results don't answer the question, say that the knowledge base
  doesn't cover it. Do not guess.
- When you use the knowledge base, end your answer with the titles of the documents you used, like:
  Sources: <title>, <title>"""

config = types.GenerateContentConfig(
    system_instruction=SYSTEM_INSTRUCTION,
    tools=[types.Tool(function_declarations=TOOL_DECLARATIONS)],
)

print("Tool and system instruction ready.")

## 10. Let the agent decide when to retrieve

Before building the full agent, let's look at **just the decision**. We send two questions and check whether Gemini asks for the tool. We don't run anything yet.

In [ ]:
def show_decision(question):
    response = client.models.generate_content(model=MODEL, contents=question, config=config)
    calls = response.function_calls or []
    print("Question:", question)
    if calls:
        for call in calls:
            print("  Decision: SEARCH  ->", call.name, dict(call.args))
    else:
        print("  Decision: ANSWER DIRECTLY")
    print()


show_decision("What are the three stopping rules for an agent loop?")
show_decision("Hi there! How are you today?")

The first question is about a course concept, so the agent asks to search, and it writes its **own** search query. The greeting needs no knowledge, so it answers directly.

The agent made both decisions using only the tool description and the system instruction.

## 11. Return retrieved information and generate the final answer

Now we build the full agent. It uses the loop from Lesson 4:

```text
Question
   ↓
Gemini decides ──── no tool call ────→ Final answer
   │
   └─ tool call: knowledge_search("...")
         ↓
      Python runs the search
         ↓
      Results go back to Gemini as a function response
         ↓
      Gemini decides again (answer, or search again)
```

Three details matter:

1. We add Gemini's turn (including its function call) to `contents`, so it remembers what it asked for.
2. We return the search results with `types.Part.from_function_response(...)`, the same way as Week 1.
3. A **step limit** (`MAX_STEPS`) stops the loop if something goes wrong.

We also add a small retry helper, because Gemini sometimes returns a temporary "busy" error.

In [ ]:
import time

MAX_STEPS = 5


def generate_with_retry(contents, config, max_retries=4):
    # Call Gemini, retrying when the service is busy (503) or rate-limited (429).
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            busy = any(code in str(e) for code in ("503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED"))
            if busy and attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Gemini is busy. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_rag_agent(question, verbose=True):
    # Ask the agent a question. It decides whether to search, then answers.
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=question)])]
    searches, sources = [], []

    for step in range(1, MAX_STEPS + 1):
        response = generate_with_retry(contents, config)
        if not response.candidates or response.candidates[0].content is None:
            return {"question": question, "answer": "The model returned no answer.",
                    "used_retrieval": bool(searches), "searches": searches, "sources": sources}

        # Keep Gemini's turn (including any function calls) in the conversation.
        contents.append(response.candidates[0].content)

        function_calls = response.function_calls or []

        # No tool calls means Gemini has its final answer.
        if not function_calls:
            return {"question": question, "answer": response.text,
                    "used_retrieval": bool(searches), "searches": searches, "sources": sources}

        response_parts = []
        for call in function_calls:
            tool_name = call.name
            tool_args = dict(call.args)
            if verbose:
                print(f"  [step {step}] Agent calls {tool_name}({tool_args})")

            # Run the tool (or return an error if the name is unknown).
            if tool_name in AVAILABLE_TOOLS:
                tool_result = AVAILABLE_TOOLS[tool_name](**tool_args)
            else:
                tool_result = {"error": f"There is no tool called {tool_name}"}

            # Record what was searched and which documents came back.
            if tool_name == "knowledge_search":
                searches.append(tool_args.get("query", ""))
                found = tool_result.get("results", [])
                sources += [r["id"] for r in found if r["id"] not in sources]
                if verbose:
                    print(f"  [step {step}] Found: {[r['title'] for r in found] or 'nothing relevant'}")

            # Send the result back to Gemini as a function response.
            response_parts.append(types.Part.from_function_response(name=tool_name, response=tool_result))

        contents.append(types.Content(role="user", parts=response_parts))

    return {"question": question, "answer": "Stopped: step limit reached before a final answer.",
            "used_retrieval": bool(searches), "searches": searches, "sources": sources}

In [ ]:
result = run_rag_agent("What are the three stopping rules for an agent loop?")

print("\nAnswer:\n")
print(result["answer"])

You should see the agent search, find **The ReAct loop and stopping rules**, and answer with that document named as its source.

## 12. Test the agent

A good RAG agent has to get **three kinds** of question right:

| Type | What should happen |
|---|---|
| Covered by the knowledge base | Search, then answer with sources |
| Doesn't need knowledge (greetings, maths) | Answer directly without searching |
| About the topic, but **not** in the knowledge base | Search, find nothing useful, and say so honestly |

The cell below runs six test questions and checks the agent's retrieval decision for each.

In [ ]:
TEST_QUESTIONS = [
    {"question": "What are the three stopping rules for an agent loop?", "should_retrieve": True},
    {"question": "How is RAG different from fine-tuning?", "should_retrieve": True},
    {"question": "Hi! What can you help me with?", "should_retrieve": False},
    {"question": "What is 18 * 24?", "should_retrieve": False},
    {"question": "How do embeddings help an agent remember more than its context window?", "should_retrieve": True},
    {"question": "What does the knowledge base say about evaluating agents with benchmarks?", "should_retrieve": True},
]

test_results = []

for i, test in enumerate(TEST_QUESTIONS, 1):
    print(f"\n[{i}] {test['question']}")
    result = run_rag_agent(test["question"])
    result["retrieval_decision_ok"] = result["used_retrieval"] == test["should_retrieve"]
    test_results.append(result)
    print("  Answer:", result["answer"])
    print("  Retrieval decision:", "OK" if result["retrieval_decision_ok"] else "CHECK THIS")

correct = sum(r["retrieval_decision_ok"] for r in test_results)
print(f"\nRetrieval decisions as expected: {correct}/{len(test_results)}")

### What to look for

- **Question 5** has two parts (embeddings *and* memory). Did the agent search once or twice? Did it combine two documents?
- **Question 6** is about agents, but the knowledge base has no document on benchmarks. Did the agent admit that, or did it guess?
- Did every knowledge-base answer end with **Sources**?

The model can decide differently on different runs. If a decision looks wrong, read the agent's search query in the output. It usually explains why.

## 13. Challenge

### Part A: Grow the knowledge base

1. Write **two new documents** on topics the knowledge base doesn't cover yet, for example *Evaluating agents*, *Prompt engineering for agents* or *Planning in agents*. Keep each one to 3–5 sentences.
2. Add them to `NEW_DOCUMENTS` below and run the cell to rebuild the knowledge base.
3. Update the tool description in Section 9 to mention your new topics, then re-run Section 9.
4. Write **three test questions**:
   - one answered by a new document,
   - one that needs a new document **and** an original one,
   - one the knowledge base still can't answer.

### Part B: Tune the threshold

Run the same question with `MIN_SIMILARITY` set to `0.1`, `0.35` and `0.6`. What changes in the search results and in the answer? Which value would you choose, and why?

### Stretch

Add a `topic` field to each document's metadata, and change `knowledge_search` to accept an optional topic filter using ChromaDB's `where={"topic": ...}` argument.

In [ ]:
# Part A: write your two documents here.
NEW_DOCUMENTS = [
    {
        "id": "kb-09",
        "title": "TODO: your first title",
        "text": "TODO: 3-5 sentences.",
    },
    {
        "id": "kb-10",
        "title": "TODO: your second title",
        "text": "TODO: 3-5 sentences.",
    },
]

if any("TODO" in doc["text"] for doc in NEW_DOCUMENTS):
    print("Write your two documents first, then run this cell again.")
else:
    collection = build_collection(KNOWLEDGE_BASE + NEW_DOCUMENTS)
    print("Knowledge base rebuilt:", collection.count(), "documents.")
    print("Now update the tool description in Section 9 and re-run it.")

In [ ]:
# Part A: your three test questions.
MY_QUESTIONS = [
    "TODO: a question answered by one of your new documents",
    "TODO: a question that needs a new document and an original one",
    "TODO: a question the knowledge base still can't answer",
]

for question in MY_QUESTIONS:
    if question.startswith("TODO"):
        print("Write your questions first.")
        break
    print(f"\n{question}")
    print("  Answer:", run_rag_agent(question)["answer"])

In [ ]:
# Part B: threshold experiment. Change the question if you like.
question = "How can agents avoid dangerous actions?"

for threshold in [0.1, 0.35, 0.6]:
    MIN_SIMILARITY = threshold
    found = knowledge_search(question).get("results", [])
    print(f"MIN_SIMILARITY = {threshold}: {[(r['title'], r['similarity']) for r in found] or 'no results'}")

MIN_SIMILARITY = 0.35   # put it back

## Recap

| Step | What you built |
|---|---|
| Embeddings | Text → numbers that capture meaning |
| ChromaDB | A store that finds the closest embeddings to a query |
| `knowledge_search()` | A retrieval tool that returns relevant documents, or an honest "nothing found" |
| Tool declaration | A description that tells Gemini **when** to retrieve |
| `run_rag_agent()` | An agent that decides, searches, reads the results and answers with sources |

**Next:** complete the starter file `lesson_05_rag_agent.py`, run it, and submit your work. See `README.md` for the submission checklist.